In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_XLA_FLAGS'] = '--tf_xla_cpu_global_jit=false --tf_xla_auto_jit=0'
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/home/alex/miniconda3/envs/jupyterproject-project'
import tensorflow as tf
# os.environ['XLA_FLAGS'] = '--tf_xla_auto_jit=-1'
# os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=-1'
tf.config.optimizer.set_jit(False)

# Configure GPU memory growth to prevent some JIT/CUDA allocation errors
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

import pandas as pd
import numpy as np
import optuna
import warnings
import logging
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.losses import Huber
from tensorflow.keras.regularizers import l2

warnings.filterwarnings('ignore', category=FutureWarning)
optuna.logging.set_verbosity(logging.WARNING)

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
if len(tf.config.list_physical_devices('GPU')) > 0:
    print("TensorFlow will use the GPU.")
else:
    print("No GPU detected. Check your drivers and CUDA installation.")

# ==========================================
# 1. LOAD, FILTER, AND PREPARE DATA
# ==========================================
# I'm using the exact same data loading pipeline as my per-product LSTM,
# but this time I'll melt to long format so ONE model sees ALL products.
print("Loading and filtering data...")

data_dir = '../preprocesing_data/processed_csv'
sales_path = os.path.join(data_dir, 'sales_data_preprocessed.csv')
weather_path = os.path.join(data_dir, 'weather_data_hourly.csv')
holidays_path = os.path.join(data_dir, 'holidays_data_preprocessed.csv')
events_path = os.path.join(data_dir, 'aberdeen_events_master_timeline.csv')

# 1.1 Load and Aggregate Sales Data (Hourly to Daily)
sales_df = pd.read_csv(sales_path)
sales_df['Date'] = pd.to_datetime(sales_df['Date']).dt.normalize()
if 'Time' in sales_df.columns:
    sales_df = sales_df.drop(columns=['Time'])
daily_sales = sales_df.groupby('Date').sum().reset_index()

# AUTO-DETECT ALL PRODUCT COLUMNS
non_product_cols = ['Date', 'Time', 'date', 'Date_Norm']
product_columns = [col for col in daily_sales.columns 
                   if col not in non_product_cols 
                   and daily_sales[col].dtype.kind in 'iufc']
print(f"Detected {len(product_columns)} products")

# 1.2 Load Weather Data
weather_df = pd.read_csv(weather_path)
weather_df['Date'] = pd.to_datetime(weather_df['Date'])

def create_weather_flags(df):
    if 'weather_code' not in df.columns:
        return df
    df['is_clear'] = (df['weather_code'] == 0).astype(int)
    df['is_cloudy'] = df['weather_code'].isin([1, 2, 3, 45, 48]).astype(int)
    df['is_rain'] = df['weather_code'].isin([51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82, 95, 96, 99]).astype(int)
    df['is_snow'] = df['weather_code'].isin([71, 73, 75, 77, 85, 86]).astype(int)
    return df.drop(columns=['weather_code'])

weather_df = create_weather_flags(weather_df)
weather_df['Date_Only'] = weather_df['Date'].dt.normalize()

# Proper daily aggregation - mean for continuous, max for binary flags
flag_weather_cols = [c for c in weather_df.columns if c.startswith('is_')]
numeric_weather_cols = [c for c in weather_df.select_dtypes(include=[np.number]).columns if c not in flag_weather_cols]

daily_weather_numeric = weather_df.groupby('Date_Only')[numeric_weather_cols].mean().reset_index()
daily_weather_flags = weather_df.groupby('Date_Only')[flag_weather_cols].max().reset_index()
daily_weather = daily_weather_numeric.merge(daily_weather_flags, on='Date_Only')
daily_weather = daily_weather.rename(columns={'Date_Only': 'Date'})

# 1.3 Load Holidays & Events
holidays_df = pd.read_csv(holidays_path)
holidays_df['Date'] = pd.to_datetime(holidays_df['Date']).dt.normalize()
daily_holidays = holidays_df.groupby('Date').max().reset_index()

events_df = pd.read_csv(events_path)
events_df['Date'] = pd.to_datetime(events_df['Date']).dt.normalize()
daily_events = events_df.groupby('Date').sum().reset_index()

# ==========================================
# ADD CYCLICAL TIME FEATURES
# ==========================================
daily_sales['Day_of_Week'] = daily_sales['Date'].dt.dayofweek
daily_sales['Month'] = daily_sales['Date'].dt.month
daily_sales['Is_Weekend'] = daily_sales['Day_of_Week'].isin([5, 6]).astype(int)
daily_sales['day_sin'] = np.sin(2 * np.pi * daily_sales['Day_of_Week'] / 7)
daily_sales['day_cos'] = np.cos(2 * np.pi * daily_sales['Day_of_Week'] / 7)
daily_sales['month_sin'] = np.sin(2 * np.pi * (daily_sales['Month'] - 1) / 12)
daily_sales['month_cos'] = np.cos(2 * np.pi * (daily_sales['Month'] - 1) / 12)
daily_sales = daily_sales.drop(columns=['Day_of_Week', 'Month'])

time_features = ['day_sin', 'day_cos', 'month_sin', 'month_cos', 'Is_Weekend']

# ==========================================
# MERGE EXTERNAL FEATURES
# ==========================================
exclude_cols = ['Date', 'Time', 'date', 'Date_Norm']
weather_features = [col for col in daily_weather.columns if col not in exclude_cols]
holiday_features = [col for col in daily_holidays.columns if col not in exclude_cols]
event_features = [col for col in daily_events.columns if col not in exclude_cols]

df_wide = daily_sales.merge(daily_weather[['Date'] + weather_features], on='Date', how='left')
df_wide = df_wide.merge(daily_holidays[['Date'] + holiday_features], on='Date', how='left')
df_wide = df_wide.merge(daily_events[['Date'] + event_features], on='Date', how='left')

num_cols = df_wide.select_dtypes(include=[np.number]).columns
df_wide[num_cols] = df_wide[num_cols].fillna(0)

flag_cols = [col for col in df_wide.columns if col.startswith('is_')]
if flag_cols:
    df_wide[flag_cols] = df_wide[flag_cols].clip(upper=1)

# Base features (shared across all products) — same as my per-product LSTM
base_feature_cols = weather_features + holiday_features + event_features + time_features
base_feature_cols = [col for col in base_feature_cols 
                     if col in df_wide.columns and df_wide[col].dtype.kind in 'iufc']

print(f"Base features (shared): {len(base_feature_cols)}")
print(f"Products to forecast: {len(product_columns)}")

# ==========================================
# PRODUCTS TO FORECAST
# ==========================================
# These are the specific menu items I want to predict
PRODUCTS_TO_FORECAST = [
 'Avo & Hal Muffin',
 'Avo, Egg & Bacon',
 'Avo, Feta & Tom',
 'Avocado on Toast',
 'Bacon',
 'Bacon Bap',
 'Bacon Egg Brioch',
 'Bacon Waffle',
 'Baked Beans',
 'Baked Beans JP',
 'Bean Soldiers',
 'Big Breakfast',
 'Black Pudding',
 'Breakfast Hash',
 'Breakfast Muffin',
 'Breakfast Wrap',
 'Buttd Mushrooms',
 'Cheese & Bean JP',
 'Cheese JP',
 'Chick Flatbread',
 'Chicken Club',
 'Chilli Carne JP',
 'Egg Bacon Brioch',
 'Egg Bap',
 'Extra Beans',
 'F.Eggs on Toast',
 'Festive Stack',
 'Fried Egg',
 'Hash Brown',
 'Hash Brown Bites',
 'Little Avo Toast',
 'Little Bean Toas',
 'Little Egg Toast',
 'Ltle Bfast Bacon',
 'Ltle Bfast Saus',
 'Mini Hash Browns',
 'P.Eggs on Toast',
 'Poached Egg',
 'Posh Beans',
 'Roll & Butter ',
 'S.Eggs on Toast',
 'Sausage',
 'Sausage Bap',
 'Scrambled Egg',
 'Streaky Bacon',
 'Tattie Scone',
 'The Breakfast',
 'Toasted Teacake',
 'Tuna JP',
 'Tuna Mayo Mix',
 'Tuna Melt Panini',
 'Tuna Panini',
 'Tuna Toastie',
 'Veg Sausage Bap',
 'Vegan Breakfast',
 'Vegan Sausage',
 'Veggie Bap',
 'Veggie Breakfast',
 'Bakery',
 'White Toast Bread',
 'Brown Toast Bread',
 'Porridge',
 'Sourdough Toast Bread',
 'Multiseed Toast Bread',
]


/home/alex/miniconda3/envs/jupyterproject-project/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Num GPUs Available:  1
TensorFlow will use the GPU.
Loading and filtering data...
Detected 222 products
Base features (shared): 25
Products to forecast: 222


In [2]:
# ==========================================
# 2. MELT TO LONG FORMAT & ADD PRODUCT-SPECIFIC FEATURES
# ==========================================
# This is the key difference from my per-product LSTM: instead of building
# a separate dataset for each product, I melt everything into one long DataFrame
# where each row is (Date, Product, Sales). This way ONE model learns from ALL
# products simultaneously — low-volume items benefit from patterns in high-volume ones.

SEQUENCE_LENGTH = 30

train_end = pd.to_datetime('2025-10-01')
val_end = pd.to_datetime('2025-11-01')
test_end = pd.to_datetime('2025-11-30')

print("Melting to long format and building lag features per product...")

# Melt: wide (one col per product) -> long (one row per product per day)
df_long = pd.melt(
    df_wide, 
    id_vars=['Date'] + base_feature_cols,
    value_vars=product_columns,
    var_name='Product_Name', 
    value_name='Sales'
)
df_long['Sales'] = df_long['Sales'].clip(lower=0)
df_long = df_long.sort_values(['Product_Name', 'Date']).reset_index(drop=True)

# Encode product as a numeric feature so the LSTM can distinguish between products
# This is the equivalent of XGBoost's categorical Product_Name column
product_encoder = LabelEncoder()
df_long['product_id'] = product_encoder.fit_transform(df_long['Product_Name'])
n_products = len(product_encoder.classes_)

# Add product-specific lag features (grouped by product, same lags as my per-product LSTM)
df_long['sales_lag_1'] = df_long.groupby('Product_Name')['Sales'].shift(1)
df_long['sales_lag_7'] = df_long.groupby('Product_Name')['Sales'].shift(7)
df_long['sales_rolling_7_mean'] = (
    df_long.groupby('Product_Name')['Sales']
    .shift(1)
    .groupby(df_long['Product_Name'])
    .transform(lambda x: x.rolling(window=7, min_periods=1).mean())
)
df_long['sales_rolling_7_std'] = (
    df_long.groupby('Product_Name')['Sales']
    .shift(1)
    .groupby(df_long['Product_Name'])
    .transform(lambda x: x.rolling(window=7, min_periods=1).std())
).fillna(0)
df_long['sales_diff_1'] = df_long.groupby('Product_Name')['Sales'].diff(1)

lag_features = ['sales_lag_1', 'sales_lag_7', 'sales_rolling_7_mean', 'sales_rolling_7_std', 'sales_diff_1']
df_long = df_long.dropna(subset=lag_features).reset_index(drop=True)

# My full feature set: base features + product ID + lag features
# The product_id lets the LSTM learn product-specific baselines
feature_cols = base_feature_cols + ['product_id'] + lag_features

print(f"Long format shape: {df_long.shape}")
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Products: {n_products}")
print(f"Date range: {df_long['Date'].min()} to {df_long['Date'].max()}")

Melting to long format and building lag features per product...
Long format shape: (322788, 34)
Features (31): ['apparent_temperature', 'precipitation', 'snowfall', 'snow_depth', 'relative_humidity_2m', 'cloud_cover', 'visibility', 'wind_speed_10m', 'wind_gusts_10m', 'is_clear', 'is_cloudy', 'is_rain', 'is_snow', 'is_holiday', 'holiday_importance', 'Days_Until_Next_Holiday', 'is_festival', 'festival_importance', 'is_match_day', 'match_importance', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'Is_Weekend', 'product_id', 'sales_lag_1', 'sales_lag_7', 'sales_rolling_7_mean', 'sales_rolling_7_std', 'sales_diff_1']
Products: 222
Date range: 2022-01-08 00:00:00 to 2025-12-31 00:00:00


In [3]:
# ==========================================
# 2B. DATA-DRIVEN PRODUCT CLUSTERING (FOR OPTUNA ONLY)
# ==========================================
#
# The final model is still ONE global LSTM trained on ALL products.
# Clustering is used ONLY to speed up Optuna:
#   1. Cluster products by demand shape (same method as Prophet)
#   2. Run Optuna on each cluster's subset (fast — ~1/4 data each)
#   3. Pick the cluster whose params scored best on validation
#   4. Train the single global model with those winning params
#
# This gives us the speed benefit of clustering without changing
# the model architecture — still one model, still comparable.

from sklearn.preprocessing import StandardScaler as ClusterScaler
from sklearn.cluster import KMeans

print("Building product clusters (for Optuna speedup only)...")

train_wide = df_wide[df_wide['Date'] <= train_end].copy()
train_wide['dow'] = train_wide['Date'].dt.dayofweek

product_features = []
for product in PRODUCTS_TO_FORECAST:
    if product not in train_wide.columns:
        continue
    series = train_wide[product].values.astype(float)
    mean_vol = np.mean(series)
    cv = np.std(series) / mean_vol if mean_vol > 0 else 999.0
    zero_frac = np.mean(series == 0)
    weekend_mask = train_wide['dow'].isin([5, 6])
    wkend_mean = train_wide.loc[weekend_mask, product].mean()
    wkday_mean = train_wide.loc[~weekend_mask, product].mean()
    weekend_ratio = wkend_mean / wkday_mean if wkday_mean > 0 else 1.0
    if 'apparent_temperature' in train_wide.columns:
        temp_corr = train_wide[[product, 'apparent_temperature']].corr().iloc[0, 1]
        temp_corr = 0.0 if np.isnan(temp_corr) else temp_corr
    else:
        temp_corr = 0.0
    product_features.append({
        'product': product, 'mean_daily_volume': mean_vol,
        'coeff_of_variation': cv, 'zero_day_fraction': zero_frac,
        'weekend_ratio': weekend_ratio, 'temp_correlation': temp_corr
    })

cluster_features_df = pd.DataFrame(product_features)

N_CLUSTERS = 4
X_cluster = cluster_features_df[['mean_daily_volume', 'coeff_of_variation',
    'zero_day_fraction', 'weekend_ratio', 'temp_correlation']].values
X_cluster_scaled = ClusterScaler().fit_transform(X_cluster)
cluster_features_df['cluster_id'] = KMeans(
    n_clusters=N_CLUSTERS, random_state=42, n_init=10
).fit_predict(X_cluster_scaled)

CLUSTER_PRODUCTS = {}
CLUSTER_HEROES = {}
for cid in range(N_CLUSTERS):
    cp = cluster_features_df[cluster_features_df['cluster_id'] == cid]
    CLUSTER_HEROES[cid] = cp.sort_values('mean_daily_volume', ascending=False).iloc[0]['product']
    CLUSTER_PRODUCTS[cid] = cp['product'].tolist()

for cid in range(N_CLUSTERS):
    print(f"  Cluster {cid} ({len(CLUSTER_PRODUCTS[cid])} products), "
          f"hero={CLUSTER_HEROES[cid]}: {', '.join(CLUSTER_PRODUCTS[cid][:6])}...")
print(f"\nClusters built. These are used ONLY for fast Optuna search.")
print(f"The final model will still be ONE global LSTM trained on ALL products.")


Building product clusters (for Optuna speedup only)...
  Cluster 0 (37 products), hero=Bacon Waffle: Avo & Hal Muffin, Avo, Egg & Bacon, Avo, Feta & Tom, Bacon Egg Brioch, Bacon Waffle, Baked Beans JP...
  Cluster 1 (24 products), hero=Bakery: Avocado on Toast, Bacon, Bacon Bap, Baked Beans, Big Breakfast, Black Pudding...
  Cluster 2 (2 products), hero=Festive Stack: Festive Stack, Tuna Mayo Mix...
  Cluster 3 (1 products), hero=Tuna Melt Panini: Tuna Melt Panini...

Clusters built. These are used ONLY for fast Optuna search.
The final model will still be ONE global LSTM trained on ALL products.


In [4]:
# ==========================================
# 3. BUILD SEQUENCES FOR GLOBAL MODEL
# ==========================================
# Each sequence is 30 days of features for a SINGLE product.
# But ALL products' sequences go into the same training set,
# so the model sees Bacon Bap patterns AND Vegan Wrap patterns.
# The product_id feature tells it which product it's looking at.

print("Building sequences for the global model...")

# Split by date (same cutoffs as my per-product LSTM)
train_data = df_long[df_long['Date'] <= train_end].copy()
val_data = df_long[(df_long['Date'] > train_end) & (df_long['Date'] <= val_end)].copy()
test_data = df_long[(df_long['Date'] > val_end) & (df_long['Date'] <= test_end)].copy()

print(f"Train: {len(train_data)} rows, Val: {len(val_data)} rows, Test: {len(test_data)} rows")

# Fit scalers on training data ONLY (same approach as before, just across all products)
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()
feature_scaler.fit(train_data[feature_cols])
target_scaler.fit(train_data[['Sales']])

def build_product_sequences(product_df, feature_cols, seq_len, feature_scaler, target_scaler):
    """Build sequences for a single product using the GLOBAL scalers.
    
    This is the same windowing logic as my per-product LSTM, but the scalers
    are fitted across ALL products, giving the model a consistent scale.
    """
    if len(product_df) < seq_len + 1:
        return None, None, None
    
    scaled_features = feature_scaler.transform(product_df[feature_cols])
    scaled_target = target_scaler.transform(product_df[['Sales']]).flatten()
    dates = product_df['Date'].values
    
    X, y, d = [], [], []
    for i in range(len(scaled_features) - seq_len):
        X.append(scaled_features[i : i + seq_len])
        y.append(scaled_target[i + seq_len])
        d.append(dates[i + seq_len])
    
    return np.array(X), np.array(y), pd.to_datetime(d)

# Build sequences for each product, then combine into one big training set
all_X_train, all_y_train = [], []
all_X_val, all_y_val = [], []
# For test, I keep track of which product each sequence belongs to
test_sequences = {}  # product_name -> (X_test, y_test, dates)

for product_name in product_columns:
    product_df = df_long[df_long['Product_Name'] == product_name].sort_values('Date').reset_index(drop=True)
    
    X_seqs, y_seqs, d_seqs = build_product_sequences(
        product_df, feature_cols, SEQUENCE_LENGTH, feature_scaler, target_scaler
    )
    
    if X_seqs is None:
        continue
    
    # Split sequences by date into train/val/test
    train_mask = d_seqs <= train_end
    val_mask = (d_seqs > train_end) & (d_seqs <= val_end)
    test_mask = (d_seqs > val_end) & (d_seqs <= test_end)
    
    if train_mask.sum() > 0:
        all_X_train.append(X_seqs[train_mask])
        all_y_train.append(y_seqs[train_mask])
    
    if val_mask.sum() > 0:
        all_X_val.append(X_seqs[val_mask])
        all_y_val.append(y_seqs[val_mask])
    
    if test_mask.sum() > 0:
        test_sequences[product_name] = {
            'X_test': X_seqs[test_mask],
            'y_test': y_seqs[test_mask],
            'dates': d_seqs[test_mask]
        }

# Stack all products into one big array — this is what the global model trains on
X_train = np.vstack(all_X_train)
y_train = np.concatenate(all_y_train)
X_val = np.vstack(all_X_val)
y_val = np.concatenate(all_y_val)

# Shuffle training data so the model doesn't see all of one product in a row
shuffle_idx = np.random.permutation(len(X_train))
X_train = X_train[shuffle_idx]
y_train = y_train[shuffle_idx]

print(f"\nGlobal training set: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Global validation set: X_val={X_val.shape}, y_val={y_val.shape}")
print(f"Products with test sequences: {len(test_sequences)}")

Building sequences for the global model...
Train: 302586 rows, Val: 6882 rows, Test: 6438 rows

Global training set: X_train=(295926, 30, 31), y_train=(295926,)
Global validation set: X_val=(6882, 30, 31), y_val=(6882,)
Products with test sequences: 222


In [5]:
# ==========================================
# 4. FAST OPTUNA TUNING VIA HERO PRODUCTS
# ==========================================
#
# Strategy (mirrors Prophet exactly):
#   1. For each cluster, run Optuna on ONLY the hero product's sequences
#      (the highest-volume product in the cluster — cleanest signal)
#   2. Evaluate each cluster's best params on the FULL validation set
#   3. Pick the params that scored best globally
#   4. Train ONE global model on ALL data with those params
#
# This is dramatically faster than tuning on all products:
#   - Original: 30 trials × all 74 products' sequences = SLOW
#   - Now: 10 trials × 1 hero product per cluster = FAST
#
# The final model still trains on ALL products — Optuna just needs
# good hyperparameters, and the hero gives the cleanest search signal.

OPTUNA_TRIALS_PER_CLUSTER = 10

print(f"Running Optuna on {N_CLUSTERS} hero products × {OPTUNA_TRIALS_PER_CLUSTER} trials each")
print(f"Then picking the best params and training ONE global model\n")

# --- Build hero-only training subsets (for Optuna only) ---
cluster_subsets = {}
for cid in range(N_CLUSTERS):
    hero = CLUSTER_HEROES[cid]
    if hero not in product_columns:
        print(f"  Cluster {cid}: hero {hero} not in data, skipping")
        continue
    
    hero_df = df_long[df_long['Product_Name'] == hero].sort_values('Date').reset_index(drop=True)
    X_seqs, y_seqs, d_seqs = build_product_sequences(
        hero_df, feature_cols, SEQUENCE_LENGTH, feature_scaler, target_scaler
    )
    if X_seqs is None:
        print(f"  Cluster {cid}: hero {hero} has insufficient data, skipping")
        continue
    
    train_mask = d_seqs <= train_end
    val_mask = (d_seqs > train_end) & (d_seqs <= val_end)
    
    if train_mask.sum() > 0:
        cluster_subsets[cid] = {
            'X_train': X_seqs[train_mask],
            'y_train': y_seqs[train_mask],
            'X_val': X_seqs[val_mask] if val_mask.sum() > 0 else X_val[:0],
            'y_val': y_seqs[val_mask] if val_mask.sum() > 0 else y_val[:0]
        }
        print(f"  Cluster {cid} hero={hero}: {train_mask.sum()} train, "
              f"{val_mask.sum()} val sequences")

# --- Run Optuna per cluster (hero only) ---
cluster_results = {}  # cid -> (best_params, best_rmse)

for cid in sorted(cluster_subsets.keys()):
    cd = cluster_subsets[cid]
    
    print(f"\n{'='*50}")
    print(f"  Tuning Cluster {cid} — hero only: {CLUSTER_HEROES[cid]}")
    print(f"  ({cd['X_train'].shape[0]} train sequences)")
    print(f"{'='*50}")
    
    def make_objective(Xt, yt, Xv, yv):
        _Xt, _yt, _Xv, _yv = Xt, yt, Xv, yv
        def objective(trial):
            tf.keras.backend.clear_session()
            batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256])
            lstm_1_units = trial.suggest_int('lstm_1_units', 64, 256, step=32)
            lstm_2_units = trial.suggest_int('lstm_2_units', 32, 128, step=16)
            dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.4)
            learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
            l2_reg = trial.suggest_float('l2_reg', 1e-5, 1e-2, log=True)
            model = Sequential([
                Input(shape=(_Xt.shape[1], _Xt.shape[2])),
                LSTM(lstm_1_units, activation='tanh', return_sequences=True),
                Dropout(dropout_rate),
                LSTM(lstm_2_units, activation='tanh'),
                Dropout(dropout_rate),
                Dense(64, activation='relu', kernel_regularizer=l2(l2_reg)),
                Dense(32, activation='relu'),
                Dense(1)
            ])
            model.compile(optimizer=Adam(learning_rate=learning_rate), loss=Huber())
            model.fit(_Xt, _yt, validation_data=(_Xv, _yv),
                      epochs=100, batch_size=batch_size,
                      callbacks=[
                          EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
                          ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)
                      ], verbose=0)
            val_preds = target_scaler.inverse_transform(model.predict(_Xv, verbose=0)).flatten()
            val_actual = target_scaler.inverse_transform(_yv.reshape(-1, 1)).flatten()
            return np.sqrt(mean_squared_error(val_actual, val_preds))
        return objective
    
    obj = make_objective(cd['X_train'], cd['y_train'], cd['X_val'], cd['y_val'])
    study = optuna.create_study(direction='minimize')
    study.optimize(obj, n_trials=OPTUNA_TRIALS_PER_CLUSTER)
    cluster_results[cid] = (study.best_params, study.best_value)
    print(f"  Best RMSE: {study.best_value:.4f}")

# --- Evaluate each cluster's best params on FULL validation set ---
print(f"\n{'='*50}")
print(f"  EVALUATING HERO PARAMS ON FULL VALIDATION SET")
print(f"{'='*50}")

best_global_rmse = float('inf')
best_params = None
best_cluster_id = None

for cid, (params, hero_rmse) in cluster_results.items():
    tf.keras.backend.clear_session()
    model = Sequential([
        Input(shape=(X_train.shape[1], X_train.shape[2])),
        LSTM(params['lstm_1_units'], activation='tanh', return_sequences=True),
        Dropout(params['dropout_rate']),
        LSTM(params['lstm_2_units'], activation='tanh'),
        Dropout(params['dropout_rate']),
        Dense(64, activation='relu', kernel_regularizer=l2(params['l2_reg'])),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=params['learning_rate']), loss=Huber())
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=100, batch_size=params['batch_size'],
              callbacks=[
                  EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
                  ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)
              ], verbose=0)
    val_preds = target_scaler.inverse_transform(model.predict(X_val, verbose=0)).flatten()
    val_actual = target_scaler.inverse_transform(y_val.reshape(-1, 1)).flatten()
    global_rmse = np.sqrt(mean_squared_error(val_actual, val_preds))
    
    print(f"  Cluster {cid} ({CLUSTER_HEROES[cid]}) params → global val RMSE: {global_rmse:.4f} "
          f"(hero val RMSE was {hero_rmse:.4f})")
    
    if global_rmse < best_global_rmse:
        best_global_rmse = global_rmse
        best_params = params
        best_cluster_id = cid

print(f"\n  WINNER: Cluster {best_cluster_id} ({CLUSTER_HEROES[best_cluster_id]}) params")
print(f"  Global Validation RMSE: {best_global_rmse:.4f}")
print(f"  Params:")
for key, value in best_params.items():
    print(f"    {key}: {value}")


Running Optuna on 4 hero products × 10 trials each
Then picking the best params and training ONE global model

  Cluster 0 hero=Bacon Waffle: 1333 train, 31 val sequences
  Cluster 1 hero=Bakery: 1333 train, 31 val sequences
  Cluster 2 hero=Festive Stack: 1333 train, 31 val sequences
  Cluster 3 hero=Tuna Melt Panini: 1333 train, 31 val sequences

  Tuning Cluster 0 — hero only: Bacon Waffle
  (1333 train sequences)


I0000 00:00:1776452268.151018  215547 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9534 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080 Ti, pci bus id: 0000:2d:00.0, compute capability: 8.6


  Best RMSE: 2.7190

  Tuning Cluster 1 — hero only: Bakery
  (1333 train sequences)
  Best RMSE: 10.9288

  Tuning Cluster 2 — hero only: Festive Stack
  (1333 train sequences)
  Best RMSE: 0.0001

  Tuning Cluster 3 — hero only: Tuna Melt Panini
  (1333 train sequences)
  Best RMSE: 0.0000

  EVALUATING HERO PARAMS ON FULL VALIDATION SET
  Cluster 0 (Bacon Waffle) params → global val RMSE: 3.2392 (hero val RMSE was 2.7190)
  Cluster 1 (Bakery) params → global val RMSE: 17.9758 (hero val RMSE was 10.9288)
  Cluster 2 (Festive Stack) params → global val RMSE: 3.3316 (hero val RMSE was 0.0001)
  Cluster 3 (Tuna Melt Panini) params → global val RMSE: 3.3060 (hero val RMSE was 0.0000)

  WINNER: Cluster 0 (Bacon Waffle) params
  Global Validation RMSE: 3.2392
  Params:
    batch_size: 32
    lstm_1_units: 160
    lstm_2_units: 96
    dropout_rate: 0.2618488090568859
    learning_rate: 0.00046818749855735817
    l2_reg: 1.005276167736066e-05


In [6]:
# ==========================================
# 5. TRAIN FINAL GLOBAL MODEL
# ==========================================
# One model, trained once, used for ALL products.
# The product_id feature inside each sequence tells the model which product it's predicting.

print("Training final global LSTM model with best parameters...")

tf.keras.backend.clear_session()

global_model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(best_params.get('lstm_1_units', 128), activation='tanh', return_sequences=True),
    Dropout(best_params.get('dropout_rate', 0.2)),
    LSTM(best_params.get('lstm_2_units', 64), activation='tanh'),
    Dropout(best_params.get('dropout_rate', 0.2)),
    Dense(64, activation='relu', kernel_regularizer=l2(best_params.get('l2_reg', 0.001))),
    Dense(32, activation='relu'),
    Dense(1)
])

global_model.compile(optimizer=Adam(learning_rate=best_params.get('learning_rate', 0.001)), loss=Huber())

global_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=best_params.get('batch_size', 64),
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=0)
    ],
    verbose=1
)

print("\nGlobal model trained.")
print(f"Model input shape: {X_train.shape[1:]}")
global_model.summary()

Training final global LSTM model with best parameters...
Epoch 1/200
9248/9248 ━━━━━━━━━━━━━━━━━━━━ 153s 16ms/step - loss: 4.4575e-04 - val_loss: 7.1408e-05 - learning_rate: 4.6819e-04
Epoch 2/200
9248/9248 ━━━━━━━━━━━━━━━━━━━━ 141s 15ms/step - loss: 7.0374e-05 - val_loss: 5.6101e-05 - learning_rate: 4.6819e-04
Epoch 3/200
9248/9248 ━━━━━━━━━━━━━━━━━━━━ 143s 16ms/step - loss: 6.0699e-05 - val_loss: 5.9379e-05 - learning_rate: 4.6819e-04
Epoch 4/200
9248/9248 ━━━━━━━━━━━━━━━━━━━━ 143s 15ms/step - loss: 5.3749e-05 - val_loss: 5.0592e-05 - learning_rate: 4.6819e-04
Epoch 5/200
9248/9248 ━━━━━━━━━━━━━━━━━━━━ 143s 15ms/step - loss: 5.5514e-05 - val_loss: 8.1223e-05 - learning_rate: 4.6819e-04
Epoch 6/200
9248/9248 ━━━━━━━━━━━━━━━━━━━━ 141s 15ms/step - loss: 4.8261e-05 - val_loss: 6.9253e-05 - learning_rate: 4.6819e-04
Epoch 7/200
9248/9248 ━━━━━━━━━━━━━━━━━━━━ 151s 16ms/step - loss: 4.8494e-05 - val_loss: 5.2997e-05 - learning_rate: 4.6819e-04
Epoch 8/200
9248/9248 ━━━━━━━━━━━━━━━━━━━━ 146s

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 160)        │       122,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 160)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 96)             │        98,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 96)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 689,669 (2.63 MB)

 Trainable params: 229,889 (898.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 459,780 (1.75 MB)

In [7]:
# ==========================================
# 6. RECURSIVE FORECAST — ALL PRODUCTS (LEAK-FREE)
# ==========================================
# Fixed version: lag features are recomputed on the fly from PREDICTED
# values, not from pre-built X_test. This avoids data leakage where the
# model would otherwise see real future sales through the lag columns.
#
# How it works:
#   - We keep a sales_history buffer (last 30 real sales before the test period)
#   - Each day we predict, we append the prediction to sales_history
#   - We recompute lag_1, lag_7, rolling_7_mean, rolling_7_std, diff_1 from that buffer
#   - We scale the recomputed lags using the same feature_scaler
#   - The non-lag features (weather, holidays, time, product_id) still come from X_test
#     since those are known in advance (exogenous), not leaked

print("Running recursive forecast for ALL products using the global model...")
print("(Lag features recomputed from predictions — no data leakage)\n")

# Work out which feature indices correspond to lags
# feature_cols = base_feature_cols + ['product_id'] + lag_features
# lag_features are the LAST 5 columns
lag_feature_names = ['sales_lag_1', 'sales_lag_7', 'sales_rolling_7_mean', 'sales_rolling_7_std', 'sales_diff_1']
n_features = len(feature_cols)
n_lags = len(lag_feature_names)
lag_start_idx = n_features - n_lags  # index where lag columns start

# We need the scaler's min/scale for just the lag columns so we can
# manually scale recomputed lags the same way MinMaxScaler would
lag_scaler_min = feature_scaler.data_min_[lag_start_idx:]
lag_scaler_scale = feature_scaler.data_range_[lag_start_idx:]
lag_scaler_scale[lag_scaler_scale == 0] = 1.0  # avoid division by zero


def scale_lags(lag_values):
    """Scale raw lag values using the fitted feature_scaler's parameters."""
    return (lag_values - lag_scaler_min) / lag_scaler_scale


def recursive_forecast_no_leak(model, X_test, seq_len, target_scaler, product_name):
    """
    Recursive forecast that recomputes lag features from predictions.
    
    Instead of trusting the pre-built lag columns in X_test (which use
    real future sales), we maintain our own sales history and recompute
    lag_1, lag_7, rolling_7_mean, rolling_7_std, and diff_1 at each step.
    """
    # Get the last SEQUENCE_LENGTH days of real sales BEFORE the test period
    # These are needed to seed the lag computations (e.g. lag_7 needs 7 days of history)
    product_train_val = df_long[
        (df_long['Product_Name'] == product_name) & (df_long['Date'] <= val_end)
    ].sort_values('Date')['Sales'].values
    
    # We need at least seq_len days of history to seed; take the last seq_len
    sales_history = list(product_train_val[-seq_len:])
    
    current_seq = X_test[0].copy()
    preds_scaled = []
    
    for i in range(len(X_test)):
        # 1. Predict one day
        pred_scaled = model.predict(current_seq.reshape(1, seq_len, -1), verbose=0)[0, 0]
        preds_scaled.append(pred_scaled)
        
        # Convert prediction back to real units and add to history
        pred_real = target_scaler.inverse_transform([[pred_scaled]])[0, 0]
        pred_real = max(pred_real, 0)
        sales_history.append(pred_real)
        
        if i + 1 < len(X_test):
            # 2. Get next day's NON-lag features from X_test (weather, holidays, etc.)
            #    These are exogenous — known in advance, no leakage
            next_feats = X_test[i + 1][-1, :].copy()
            
            # 3. Recompute lag features from our sales_history (which uses predictions)
            h = sales_history  # shorthand
            lag_1 = h[-1]                                          # yesterday (predicted)
            lag_7 = h[-7] if len(h) >= 7 else h[0]               # 7 days ago
            rolling_7 = h[-7:] if len(h) >= 7 else h[:]
            rolling_7_mean = np.mean(rolling_7)
            rolling_7_std = np.std(rolling_7, ddof=0) if len(rolling_7) > 1 else 0.0
            diff_1 = h[-1] - h[-2] if len(h) >= 2 else 0.0
            
            raw_lags = np.array([lag_1, lag_7, rolling_7_mean, rolling_7_std, diff_1])
            scaled_lags = scale_lags(raw_lags)
            
            # 4. Overwrite the lag columns with our recomputed values
            next_feats[lag_start_idx:] = scaled_lags
            
            # 5. Slide the window forward
            current_seq = np.roll(current_seq, -1, axis=0)
            current_seq[-1, :] = next_feats
    
    preds_real = target_scaler.inverse_transform(np.array(preds_scaled).reshape(-1, 1)).flatten()
    preds_real = np.maximum(preds_real, 0)
    return preds_real


def calculate_horizon_metrics(df_slice, in_sample_naive_mae):
    """Calculate all metrics for a given horizon slice."""
    if len(df_slice) == 0:
        return [0] * 7
    y_true = df_slice['Sales'].values
    y_pred = df_slice['LSTM_Prediction'].values
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    smape = np.mean(numerator / (denominator + 1e-8))
    mase = mae / (in_sample_naive_mae + 1e-8)
    forecast_bias = np.mean(y_pred - y_true)
    return [mae, mse, rmse, mape, smape, mase, forecast_bias]


# ── helper: formatted product scorecard ──────────────────────
def print_product_scorecard(product_name, product_idx, total_products,
                            metrics_df, test_df):
    """Print a clean, aligned scorecard for one product."""
    w = 72  # total box width

    # ── header ──
    print(f"\n{'┌' + '─' * (w - 2) + '┐'}")
    title = f"Product {product_idx}/{total_products}: {product_name}"
    print(f"│{title:^{w - 2}}│")
    print(f"{'├' + '─' * (w - 2) + '┤'}")

    # ── metrics table ──
    horizons = list(metrics_df.columns[1:])
    metric_names = metrics_df['Metric'].tolist()

    # column widths
    lbl_w = max(len(m) for m in metric_names) + 2
    col_w = max(len(h) for h in horizons) + 2
    row_fmt = f"│ {{:<{lbl_w}}}│" + "│".join([f"{{:>{col_w}}}"] * len(horizons)) + f"│"

    # header row
    print(row_fmt.format("Metric", *horizons))
    print(f"│{'─' * (lbl_w + 1)}┼" + "┼".join(["─" * col_w] * len(horizons)) + f"│")

    for _, row in metrics_df.iterrows():
        vals = [str(row[h]) for h in horizons]
        print(row_fmt.format(row['Metric'], *vals))

    print(f"{'├' + '─' * (w - 2) + '┤'}")

    # ── reality check (first 10 days) ──
    rc = test_df[['Date_Only', 'Sales', 'LSTM_Prediction_Rounded',
                   'Mistake_Amount']].head(10).copy()
    rc.columns = ['Date', 'Actual', 'Predicted', 'Error']
    rc['Date'] = rc['Date'].astype(str)

    sub_title = "Reality Check (first 10 days)"
    print(f"│{sub_title:^{w - 2}}│")
    print(f"│{'─' * (w - 2)}│")

    d_w, a_w, p_w, e_w = 12, 8, 10, 7
    rc_fmt = f"│  {{:<{d_w}}} {{:>{a_w}}} {{:>{p_w}}} {{:>{e_w}}}  │"

    # Use fixed padding to stay inside the box
    print(rc_fmt.format("Date", "Actual", "Predicted", "Error"))
    print(f"│  {'─' * d_w} {'─' * a_w} {'─' * p_w} {'─' * e_w}  │")
    for _, r in rc.iterrows():
        print(rc_fmt.format(
            str(r['Date']),
            str(int(r['Actual'])),
            str(int(r['Predicted'])),
            str(int(r['Error']))
        ))

    print(f"{'└' + '─' * (w - 2) + '┘'}")


# ── Run forecasts for each product using the ONE global model ──
all_results = {}
all_test_dfs = {}

# Only forecast the 74 target products
products_to_predict = [p for p in sorted(test_sequences.keys()) if p in PRODUCTS_TO_FORECAST]

for product_idx, product_name in enumerate(products_to_predict, start=1):
    seq_data = test_sequences[product_name]
    
    # Run recursive forecast with leak-free lag recomputation
    preds_real = recursive_forecast_no_leak(
        global_model, seq_data['X_test'], SEQUENCE_LENGTH, target_scaler, product_name
    )
    actual_real = target_scaler.inverse_transform(seq_data['y_test'].reshape(-1, 1)).flatten()
    
    # Build test DataFrame — same structure as my per-product LSTM
    test_df = pd.DataFrame({
        'Date': seq_data['dates'],
        'Sales': actual_real,
        'LSTM_Prediction': preds_real
    })
    
    # In-sample naive MAE for MASE (from training data for this product)
    product_train = df_long[
        (df_long['Product_Name'] == product_name) & (df_long['Date'] <= train_end)
    ]['Sales'].values
    in_sample_naive_mae = np.mean(np.abs(np.diff(product_train))) if len(product_train) > 1 else 1.0
    
    test_df['Date_Only'] = test_df['Date'].dt.date
    t1 = pd.to_datetime('2025-11-02').date()
    t7 = pd.to_datetime('2025-11-08').date()
    t30 = pd.to_datetime('2025-11-30').date()
    
    df_1d = test_df[test_df['Date_Only'] == t1]
    df_1w = test_df[(test_df['Date_Only'] >= t1) & (test_df['Date_Only'] <= t7)]
    df_1m = test_df[(test_df['Date_Only'] >= t1) & (test_df['Date_Only'] <= t30)]
    
    metrics_df = pd.DataFrame({
        'Metric': ['MAE', 'MSE', 'RMSE', 'MAPE', 'SMAPE', 'MASE', 'Bias'],
        '1-Day (Nov 2)': calculate_horizon_metrics(df_1d, in_sample_naive_mae),
        '1-Week (Nov 2-8)': calculate_horizon_metrics(df_1w, in_sample_naive_mae),
        '1-Month (Nov 2-30)': calculate_horizon_metrics(df_1m, in_sample_naive_mae)
    })
    
    for col in metrics_df.columns[1:]:
        metrics_df[col] = metrics_df[col].apply(lambda x: f"{x:.4f}")
    
    test_df['LSTM_Prediction_Rounded'] = test_df['LSTM_Prediction'].round().astype(int)
    test_df['Mistake_Amount'] = (test_df['Sales'] - test_df['LSTM_Prediction_Rounded']).abs()
    
    # Print the clean scorecard
    print_product_scorecard(product_name, product_idx, len(products_to_predict),
                            metrics_df, test_df)
    
    all_results[product_name] = metrics_df
    all_test_dfs[product_name] = test_df

print(f"\n{'┌' + '─' * 70 + '┐'}")
done_msg = f"ALL PRODUCTS COMPLETE ({len(all_results)}/{len(products_to_predict)})"
print(f"│{done_msg:^70}│")
print(f"{'└' + '─' * 70 + '┘'}")


Running recursive forecast for ALL products using the global model...
(Lag features recomputed from predictions — no data leakage)


┌──────────────────────────────────────────────────────────────────────┐
│                    Product 1/64: Avo & Hal Muffin                    │
├──────────────────────────────────────────────────────────────────────┤
│ Metric │       1-Day (Nov 2)│    1-Week (Nov 2-8)│  1-Month (Nov 2-30)│
│────────┼────────────────────┼────────────────────┼────────────────────│
│ MAE    │              0.0092│              0.0021│              0.0005│
│ MSE    │              0.0001│              0.0000│              0.0000│
│ RMSE   │              0.0092│              0.0038│              0.0018│
│ MAPE   │ 41347952148480.0000│  9262461279085.7148│  2235766515641.3794│
│ SMAPE  │              2.0000│              0.8571│              0.2069│
│ MASE   │              0.0199│              0.0045│              0.0011│
│ Bias   │              0.0092│              0.0021│    

In [8]:
# ==========================================
# 7. SUMMARY TABLE ACROSS ALL PRODUCTS
# ==========================================

summary_rows = []
for product_name, metrics_df in all_results.items():
    mae_val = metrics_df.loc[metrics_df['Metric'] == 'MAE', '1-Month (Nov 2-30)'].values[0]
    rmse_val = metrics_df.loc[metrics_df['Metric'] == 'RMSE', '1-Month (Nov 2-30)'].values[0]
    mase_val = metrics_df.loc[metrics_df['Metric'] == 'MASE', '1-Month (Nov 2-30)'].values[0]
    bias_val = metrics_df.loc[metrics_df['Metric'] == 'Bias', '1-Month (Nov 2-30)'].values[0]
    summary_rows.append({'Product': product_name, 'MAE': mae_val, 'RMSE': rmse_val, 'MASE': mase_val, 'Bias': bias_val})

summary_df = pd.DataFrame(summary_rows)

# ── Pretty-print the summary table ──
w = 82
print(f"\n{'┌' + '─' * (w - 2) + '┐'}")
print(f"│{'SUMMARY: 1-Month Metrics — Global LSTM (All Products)':^{w - 2}}│")
print(f"{'├' + '─' * (w - 2) + '┤'}")

p_w = 35  # product name width
m_w = 9   # metric column width
hdr = f"│ {'Product':<{p_w}} │ {'MAE':>{m_w}} │ {'RMSE':>{m_w}} │ {'MASE':>{m_w}} │ {'Bias':>{m_w}} │"
print(hdr)
print(f"│{'─' * (p_w + 2)}┼{'─' * (m_w + 2)}┼{'─' * (m_w + 2)}┼{'─' * (m_w + 2)}┼{'─' * (m_w + 2)}│")

for _, row in summary_df.iterrows():
    name = row['Product'][:p_w]
    print(f"│ {name:<{p_w}} │ {str(row['MAE']):>{m_w}} │ {str(row['RMSE']):>{m_w}} │ {str(row['MASE']):>{m_w}} │ {str(row['Bias']):>{m_w}} │")

# ── Averages row ──
avg_mae  = summary_df['MAE'].astype(float).mean()
avg_rmse = summary_df['RMSE'].astype(float).mean()
avg_mase = summary_df['MASE'].astype(float).mean()
avg_bias = summary_df['Bias'].astype(float).mean()
print(f"│{'─' * (p_w + 2)}┼{'─' * (m_w + 2)}┼{'─' * (m_w + 2)}┼{'─' * (m_w + 2)}┼{'─' * (m_w + 2)}│")
print(f"│ {'AVERAGE':<{p_w}} │ {avg_mae:>{m_w}.4f} │ {avg_rmse:>{m_w}.4f} │ {avg_mase:>{m_w}.4f} │ {avg_bias:>{m_w}.4f} │")
print(f"{'└' + '─' * (w - 2) + '┘'}")

# ==========================================
# 8. SAVE RESULTS
# ==========================================
os.makedirs('../results', exist_ok=True)

summary_df.to_csv('../results/lstm_global_daily_all_products_summary.csv', index=False)
print(f"\nSaved summary to ../results/lstm_global_daily_all_products_summary.csv")

# Save all predictions for the comparison dashboard
all_predictions = []
for product_name, test_df in all_test_dfs.items():
    pred_df = test_df[['Date', 'Sales', 'LSTM_Prediction']].copy()
    pred_df['Product_Name'] = product_name
    all_predictions.append(pred_df)

all_predictions_df = pd.concat(all_predictions, ignore_index=True)
all_predictions_df.to_csv('../results/lstm_global_daily_all_predictions.csv', index=False)
print(f"Saved all predictions to ../results/lstm_global_daily_all_predictions.csv")

# ==========================================
# 9. SQLITE STORAGE (same schema as other models)
# ==========================================
import sqlite3
from datetime import datetime

prediction_timestamp = datetime.now()
run_id = prediction_timestamp.strftime('%Y%m%d_%H%M') + '_LSTM_Global_Daily'

database_path = '../results/model_tracking.db'
db_connection = sqlite3.connect(database_path)

with db_connection:
    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS predictions_log (
            run_id TEXT, model_type TEXT, target_date TEXT,
            prediction_made_on TEXT, product_name TEXT,
            actual_sales REAL, predicted_sales REAL
        )
    """)
    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS metrics_summary (
            run_id TEXT, model_type TEXT, product_name TEXT,
            evaluation_horizon TEXT, WAPE REAL, MASE REAL, MAE REAL, Bias REAL
        )
    """)

    # Log predictions
    predictions_for_db = all_predictions_df.rename(columns={
        'Date': 'target_date', 'Product_Name': 'product_name',
        'Sales': 'actual_sales', 'LSTM_Prediction': 'predicted_sales'
    })
    predictions_for_db['run_id'] = run_id
    predictions_for_db['model_type'] = 'LSTM_Global_Daily'
    predictions_for_db['prediction_made_on'] = str(prediction_timestamp)
    predictions_for_db['target_date'] = predictions_for_db['target_date'].astype(str)
    predictions_for_db.to_sql('predictions_log', db_connection, if_exists='append', index=False)
    print(f"  Logged {len(predictions_for_db)} predictions to predictions_log")

    # Log metrics per product per horizon
    t1 = pd.to_datetime('2025-11-02').date()
    t7 = pd.to_datetime('2025-11-08').date()
    t30 = pd.to_datetime('2025-11-30').date()

    combined_test = pd.concat([
        tdf.assign(Product_Name=pname) for pname, tdf in all_test_dfs.items()
    ], ignore_index=True)
    combined_test['Date_Only'] = pd.to_datetime(combined_test['Date']).dt.date

    metrics_rows = []
    horizons = {
        '1-Day': combined_test[combined_test['Date_Only'] == t1],
        '1-Week': combined_test[(combined_test['Date_Only'] >= t1) & (combined_test['Date_Only'] <= t7)],
        '1-Month': combined_test[(combined_test['Date_Only'] >= t1) & (combined_test['Date_Only'] <= t30)]
    }

    for horizon_label, horizon_df in horizons.items():
        if len(horizon_df) == 0:
            continue
        for product_name in all_test_dfs.keys():
            ph = horizon_df[horizon_df['Product_Name'] == product_name]
            if len(ph) > 0:
                yt = ph['Sales'].values.astype(float)
                yp = ph['LSTM_Prediction'].values.astype(float)
                pt = df_long[(df_long['Product_Name'] == product_name) & (df_long['Date'] <= train_end)]['Sales'].values
                nm = np.mean(np.abs(np.diff(pt))) if len(pt) > 1 else 1.0
                tae = np.sum(np.abs(yt - yp))
                ta = np.sum(np.abs(yt))
                metrics_rows.append({
                    'run_id': run_id, 'model_type': 'LSTM_Global_Daily',
                    'product_name': product_name, 'evaluation_horizon': horizon_label,
                    'WAPE': tae / ta if ta > 0 else np.nan,
                    'MASE': mean_absolute_error(yt, yp) / nm if nm > 0 else np.nan,
                    'MAE': mean_absolute_error(yt, yp), 'Bias': np.mean(yp - yt)
                })
        # ALL_PRODUCTS aggregate
        yt_all = horizon_df['Sales'].values.astype(float)
        yp_all = horizon_df['LSTM_Prediction'].values.astype(float)
        tae_all = np.sum(np.abs(yt_all - yp_all))
        ta_all = np.sum(np.abs(yt_all))
        mae_all = mean_absolute_error(yt_all, yp_all)
        avg_nm = np.mean([np.mean(np.abs(np.diff(df_long[(df_long['Product_Name']==p) & (df_long['Date']<=train_end)]['Sales'].values))) for p in all_test_dfs.keys() if len(df_long[(df_long['Product_Name']==p) & (df_long['Date']<=train_end)]) > 1])
        metrics_rows.append({
            'run_id': run_id, 'model_type': 'LSTM_Global_Daily',
            'product_name': 'ALL_PRODUCTS', 'evaluation_horizon': horizon_label,
            'WAPE': tae_all / ta_all if ta_all > 0 else np.nan,
            'MASE': mae_all / avg_nm if avg_nm > 0 else np.nan,
            'MAE': mae_all, 'Bias': np.mean(yp_all - yt_all)
        })

    if metrics_rows:
        pd.DataFrame(metrics_rows).to_sql('metrics_summary', db_connection, if_exists='append', index=False)
        print(f"  Logged {len(metrics_rows)} metric rows to metrics_summary")

db_connection.close()
print(f"\nRun ID: {run_id}")
print("Done!")



┌────────────────────────────────────────────────────────────────────────────────┐
│             SUMMARY: 1-Month Metrics — Global LSTM (All Products)              │
├────────────────────────────────────────────────────────────────────────────────┤
│ Product                             │       MAE │      RMSE │      MASE │      Bias │
│─────────────────────────────────────┼───────────┼───────────┼───────────┼───────────│
│ Avo & Hal Muffin                    │    0.0005 │    0.0018 │    0.0011 │    0.0005 │
│ Avo, Egg & Bacon                    │    1.8169 │    2.2727 │    3.6825 │   -0.9556 │
│ Avo, Feta & Tom                     │    1.2622 │    1.7086 │    2.6612 │   -0.2054 │
│ Avocado on Toast                    │    1.0965 │    1.3942 │    0.5215 │   -0.5140 │
│ Bacon                               │    2.0784 │    3.1421 │    0.5248 │   -1.0878 │
│ Bacon Bap                           │    4.2969 │    5.8124 │    0.8687 │   -2.0820 │
│ Bacon Egg Brioch                    │    1.0